In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn_prg import average_precision_recall_gain

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'bin').is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

DATABASE_PATH = PROJECT_DIR / 'local_data' / 'ani_microbial_eukaryotes.duckdb'
RESULTS_PATH = PROJECT_DIR / 'figs' / 'auprg_by_family_sklearn_prg.tsv'

#ANI_MIN = 75.0
#AF_MIN = 0.20
MARKER_IDENTITY_MIN = 50.0
ITS_ALIGNMENT_LENGTH_MIN = 50
FIVE_EIGHT_S_ALIGNMENT_LENGTH_MIN = 150
MIN_SPECIES_REPRESENTATIVES = 10
METRICS = ['ANI', 'AAI', 'ITS1', 'ITS2', '5.8S']


if not DATABASE_PATH.is_file():
    raise FileNotFoundError(DATABASE_PATH)


In [2]:
def load_filtered_pairs():
    con = duckdb.connect(str(DATABASE_PATH), read_only=True)
    try:
        con.execute('SET threads=4')
        con.execute('SET max_expression_depth=100000')
        return con.execute(
            f"""
            WITH retained_metadata AS (
                SELECT genome_id,
                       phylum_2026_09_01 AS phylum,
                       family_2026_09_01 AS family_name,
                       species_2026_09_01 AS species
                FROM genomes
                WHERE phylum_2026_09_01 IS NOT NULL
                  AND family_2026_09_01 IS NOT NULL
                  AND species_2026_09_01 IS NOT NULL
                  AND trim(phylum_2026_09_01) <> ''
                  AND trim(family_2026_09_01) <> ''
                  AND trim(species_2026_09_01) <> ''
            ),
            eligible_species AS (
                SELECT species
                FROM retained_metadata
                GROUP BY species
                HAVING count(*) >= {MIN_SPECIES_REPRESENTATIVES}
            ),
            retained AS (
                SELECT m.*
                FROM retained_metadata m
                JOIN eligible_species e USING (species)
            ),
            base_pairs AS (
                SELECT p.genome1_id, p.genome2_id, p.ani AS ANI, p.aai AS AAI,
                       m1.phylum, m1.family_name AS family,
                       (m1.species = m2.species)::INTEGER AS same_species
                FROM pairwise_metrics p
                JOIN retained m1 ON p.genome1_id = m1.genome_id
                JOIN retained m2 ON p.genome2_id = m2.genome_id
                WHERE p.ani IS NOT NULL
                  AND p.af IS NOT NULL
                  AND m1.phylum = m2.phylum
                  AND m1.family_name = m2.family_name
            ),
            marker_best AS (
                SELECT genome1_id, genome2_id, region, marker_identity
                FROM marker_pairwise_results
                WHERE marker_identity BETWEEN {MARKER_IDENTITY_MIN} AND 100
                  AND (
                        (region IN ('ITS1', 'ITS2')
                         AND alignment_length >= {ITS_ALIGNMENT_LENGTH_MIN})
                     OR (region = '5.8S'
                         AND alignment_length >= {FIVE_EIGHT_S_ALIGNMENT_LENGTH_MIN})
                  )
            ),
            its_wide AS (
                SELECT genome1_id, genome2_id,
                       max(marker_identity) FILTER (WHERE region = 'ITS1') AS ITS1,
                       max(marker_identity) FILTER (WHERE region = 'ITS2') AS ITS2,
                       max(marker_identity) FILTER (WHERE region = '5.8S') AS "5.8S"
                FROM marker_best
                GROUP BY genome1_id, genome2_id
            )
            SELECT p.phylum, p.family, p.same_species, p.ANI, p.AAI,
                   i.ITS1, i.ITS2, i."5.8S"
            FROM base_pairs p
            LEFT JOIN its_wide i
              USING (genome1_id, genome2_id)
            """
        ).fetchdf()
    finally:
        con.close()


In [3]:
def calculate_family_auprg(pairs):
    complete = pairs.dropna(subset=METRICS).copy()
    rows = []
    for (phylum, family), family_pairs in complete.groupby(
        ['phylum', 'family'], observed=True, sort=True
    ):
        labels = family_pairs['same_species'].to_numpy(dtype=np.int8)
        positives = int(labels.sum())
        negatives = int(labels.size - positives)
        if positives == 0 or negatives == 0:
            continue
        for metric in METRICS:
            rows.append({
                'phylum': phylum,
                'family': family,
                'metric': metric,
                'auPRG': float(average_precision_recall_gain(
                    labels, family_pairs[metric].to_numpy(dtype=float)
                )),
                'comparisons': labels.size,
            })
    results = pd.DataFrame(rows)
    if results.empty:
        raise RuntimeError('No eligible families with both comparison classes.')
    return results


def make_report_table(results):
    scores = (
        results.pivot(index=['phylum', 'family'], columns='metric', values='auPRG')
        .reindex(columns=METRICS)
    )
    comparisons = (
        results.groupby(['phylum', 'family'], observed=True)['comparisons']
        .first()
        .astype(int)
    )
    table = scores.join(comparisons).reset_index()
    table.columns.name = None
    return (
        table.rename(columns={
            'phylum': 'Phylum',
            'family': 'Family',
            'comparisons': 'Comparisons, n',
        })
        [['Phylum', 'Family', 'Comparisons, n', *METRICS]]
        .sort_values(['Phylum', 'Family'], kind='stable')
        .reset_index(drop=True)
    )


def style_report_table(table):
    display_table = table.copy()
    display_table.loc[display_table['Phylum'].duplicated(), 'Phylum'] = ''

    def bold_row_max(row):
        styles = pd.Series('', index=row.index)
        metric_values = row[METRICS]
        styles.loc[METRICS] = np.where(
            metric_values.eq(metric_values.max()), 'font-weight: bold', ''
        )
        return styles

    return (
        display_table.style
        .hide(axis='index')
        .format({'Comparisons, n': '{:,.0f}', **{metric: '{:.4f}' for metric in METRICS}})
        .apply(bold_row_max, axis=1)
    )


In [4]:
pairs = load_filtered_pairs()
results = calculate_family_auprg(pairs)
report_table = make_report_table(results)
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
report_table.to_csv(RESULTS_PATH, sep='\t', index=False, float_format='%.4f')
style_report_table(report_table)


Phylum,Family,"Comparisons, n",ANI,AAI,ITS1,ITS2,5.8S
Ascomycota,Aspergillaceae,"106,425",0.6002,0.7133,0.3661,0.4527,0.5609
,Debaryomycetaceae,"2,589",0.9997,0.9997,1.0000,0.9798,0.8733
,Nectriaceae,"2,576",0.9879,0.9892,0.8794,0.9180,0.5000
,Pichiaceae,"1,616",0.9996,0.9995,0.9950,0.9660,0.4833
,Saccharomycetaceae,"412,596",0.9508,0.9267,0.5222,0.1745,0.3404
Euglenozoa,Trypanosomatidae,379,0.3642,0.3916,0.5235,0.2735,0.4484
Oomycota,Peronosporaceae,812,0.9991,1.0000,0.6211,0.9177,0.8003
